# Reproducibility

Every random draw in the library goes through the `rng` argument of the function that
makes it. Nothing reads or changes NumPy's global random state. This tutorial shows

- the accepted kinds of `rng` and when two calls give equal results,
- what `rng=None` records,
- how to replay a computation from a result's provenance,
- that parallel evaluation does not change the numbers.

See {doc}`../user_guide/randomness`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import numpy as np

import active_subspaces as ac

piston = ac.test_functions.PISTON
X = piston.space.sample(200, rng=0)
G = ac.evaluate_grads(piston, X=X)

## Kinds of `rng`

`rng` accepts a non-negative integer, a `numpy.random.SeedSequence`, a
`numpy.random.Generator`, or `None`. An integer and the generator built from it give the
same result. `==` on results compares every field except the provenance.

In [ ]:
def subspace(rng):
    return ac.active_subspace(grads=G, space=piston.space, nboot=50, rng=rng)


a = subspace(42)
b = subspace(np.random.default_rng(42))
c = subspace(np.random.SeedSequence(42))
print(a == b, a == c)

A `Generator` is consumed: passing the same generator twice gives two different
bootstraps, as two draws from one stream should. Pass an integer (or a fresh generator)
to repeat a result.

In [ ]:
gen = np.random.default_rng(42)
first, second = subspace(gen), subspace(gen)
print("first == a:", first == a, "| second == first:", second == first)

## `rng=None` is recorded

With `rng=None` the library draws fresh entropy from the operating system. The result
differs from run to run, but the entropy is recorded in `provenance.rng`, so the run can
be repeated.

In [ ]:
fresh = subspace(None)
fresh.provenance.rng

## Replaying a result

The library has no replay function; the recipe below rebuilds the generator a function
saw on entry from `provenance.rng`. A description of kind `"seed_sequence"` holds the
seed sequence; one of kind `"generator"` also holds the generator's state at entry.
Integers are stored as decimal strings so they survive JSON.

In [ ]:
def _ints(obj):
    # Decimal strings back to integers, recursively.
    if isinstance(obj, dict):
        if "__uint_array__" in obj:
            return np.array([int(v) for v in obj["data"]], dtype=obj["__uint_array__"])
        return {k: v if k == "bit_generator" else _ints(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_ints(v) for v in obj]
    if isinstance(obj, str) and obj.lstrip("-").isdigit():
        return int(obj)
    return obj


def replay_rng(desc):
    """Rebuild the generator described by a result's provenance.rng."""
    ss = desc if desc["kind"] == "seed_sequence" else desc["seed_sequence"]
    bit_generator = getattr(np.random, desc["bit_generator"])
    if ss is None:  # a generator without a seed sequence: restore the state only
        bg = bit_generator()
    else:
        ent = ss["entropy"]
        entropy = int(ent) if isinstance(ent, str) else [int(e) for e in ent]
        seq = np.random.SeedSequence(
            entropy,
            spawn_key=tuple(ss["spawn_key"]),
            n_children_spawned=ss["n_children_spawned"],
            pool_size=ss.get("pool_size", 4),
        )
        bg = bit_generator(seq)
    if desc["kind"] == "generator":
        bg.state = _ints(desc["state"])
    return np.random.Generator(bg)

Passing the rebuilt generator with the same inputs reproduces the result exactly, both
for `rng=None` and for the second call on a shared generator above:

In [ ]:
print(subspace(replay_rng(fresh.provenance.rng)) == fresh)
print(subspace(replay_rng(second.provenance.rng)) == second)

The provenance also records digests of the input arrays, so you can check that the
inputs are the same before replaying. The digest of the gradients is that of the
physical array you passed:

In [ ]:
fresh.provenance.parameters["grads"]

## Parallel evaluation

Functions that evaluate a model or run independent tasks take an `executor` (any
`concurrent.futures.Executor`). Each task gets its own random stream, fixed by the
problem, not by the number of workers, so the numbers do not depend on the executor.

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    G_par = ac.evaluate_grads(piston, X=X, executor=pool, batch_size=25)
    a_par = ac.active_subspace(grads=G, space=piston.space, nboot=50, rng=42, executor=pool)
print(np.array_equal(G_par, G), a_par == a)

## Limits

Results are bitwise equal when the inputs and the `rng` description match, and so do
the versions of the library, NumPy, SciPy and Python, the platform, the BLAS build and
its thread count. A BLAS whose rounding depends on where arrays sit in memory can
still change the last bits of some fields between runs. Across platforms or BLAS
builds, results agree to floating-point tolerances only, and a new library version
may use the random streams differently. {doc}`../user_guide/randomness` gives the
details.